# Extract Patron Data

Extracts Patron assets and their milestones, effects buffs and targets from Anno 117
into CSV and JSON files. Output goes to `results/tables/`.

Run all cells from the project root.

In [1]:
from pathlib import Path
import json
import re

import pandas as pd

from assetextractor.extraction.utils import Config
from assetextractor.parsing.core.assets import AssetCache

## Load assets

Setup game data for processing.

In [2]:
config = Config.from_json("config.json")
assets = AssetCache.load(config)
templates = assets.templates

print(f"Total assets: {len(assets.elements)}")
print(f"Total texts: {len(assets.texts.elements)}")

Total assets: 30713
Total texts: 33147


In [3]:
asset = assets[37553] # Troop Roman Celtic Auxilia
raw_list = asset.find_value("Maintenance.Maintenances")
print("Asset 37553")
for item in raw_list:
    product = item.find_value("Product")
    print(product, product.__class__.__name__)

asset = assets[43664] # Troop Roman Murmillo Gladiator
print("Asset 43664", asset.find_value("Maintenance.Maintenances"))

Asset 37553
Denarii Asset
Wader Workforce Asset
Plebeian Workforce Asset
Equites Workforce Asset
Patrician Workforce Asset
Asset 43664 [Item (0), Item (1), Item (2), Item (3), Item (4)]


In [4]:
## Debugging datasets

# Access the datasets cache
datasets = assets.datasets

print(f"Total datasets: {len(datasets.elements)}\n")

building_type_dataset = datasets["BuildingType"].literals
print(building_type_dataset)

region_dataset = datasets["Region"].literals
print(region_dataset)

Total datasets: 556

['Other', 'Residence', 'Factory', 'Mine', 'Logistic', 'Public', 'BuildingModule', 'Warehouse']
['Meta', 'Roman', 'Celtic', 'Egyptian']


## Extract the Patron assets

Extract the patrons from `Patron` (8 assets).

In [5]:
# Misc: This allow reloading .py modules into jupyter. Only those marked with %aimport will be reloaded.
%load_ext autoreload
%autoreload 3

In [6]:
from assetextractor.conversion.statistics.patron_extractor import PatronExtractor

extractor = PatronExtractor(assets)
# extractor_de = PatronExtractor(assets, language="german")
patrons_data = extractor.extract_all()
print(f"Processed {len(patrons_data.keys())} patrons")

Processed 8 patrons


## Print Patrons and their details

Use `extractor.print_patrons()` to see everything, or `extractor.print_patrons(guid=80562)` (Mars as example) to inspect a specific one.

In [7]:
extractor.print_patrons(43594) # Ceres


                                 PATRON: PatronCeres (GUID: 43594)                                  
Shrine: Asset Pool All Ceres Shrines (GUID: 82494)
Name: Shrine of Ceres
----------------------------------------------------------------------------------------------------
Veneration Effect: Vervactor's Plough (GUID: 43620)
All Farms can support up to 50% more field modules, allowing an increase of up to 50% productivity.
----------------------------------------------------------------------------------------------------
Exaltation Effect: Conditor's Grace (GUID: 43603)
Increases the storage limit on all islands by 300t.
Portraits 
- Big: artwork_deity_ceres_1248_0
- Small: artwork_deity_ceres_small_88_0
Title:       Ceres Augusta
Description: Increased production of Porridge, Bread, Wine, Olive Oil, Beer, Writing Tablets
Asset GUID:  43598
----------------------------------------------------------------------------------------------------
Buffs: 1
  |- 1 Buff - Ceres Buff Production

### Print Affected Production Chains

We can also print other details like the affected chains.

In [8]:
# mars_guid = 80562
# mars_patron = patrons_data[mars_guid]

vulcan_guid = 144800
vulcan_patron = patrons_data[vulcan_guid]

print(f"{'*' * 100}")
print(f"Patron: {vulcan_patron.text} (GUID: {vulcan_guid})")
print(f"Canonical Name: {vulcan_patron.canonical_name}")
print(f"Icon Canonical Name: {vulcan_patron.icon.canonical_name}")

# Showcase affected chains.
print(f"{'*' * 100}")
print(f"Affected Production Chains")
print(f"{'*' * 100}")

affected_chains = vulcan_patron.production_chains_by_target

# Top level key: Production Chain Asset
for chain, targets in affected_chains.items():
    print(f"Chain GUID: {chain.guid} -> Text: {chain.text}")
    print(f"  Affects {len(targets)} total target assets:")
    
    # Inner level key: Target asset GUID
    for target_guid, target_asset in targets.items():
        print(f"    |- Target Asset: {target_asset.name} (GUID: {target_guid})")
    print("-" * 100)

****************************************************************************************************
Patron: Vulcan (GUID: 144800)
Canonical Name: patron_vulcan
Icon Canonical Name: icon_patron_vulcan
****************************************************************************************************
Affected Production Chains
****************************************************************************************************
Chain GUID: 50225 -> Text: Mines
  Affects 8 total target assets:
    |- Target Asset: Production Mountain Celtic Copper Ore (GUID: 5290)
    |- Target Asset: Production Mountain Roman Celtic Silver Ore (GUID: 5980)
    |- Target Asset: Production Mountain Celtic Tin Ore (GUID: 5291)
    |- Target Asset: Production Mountain Roman Iron Ore (GUID: 2918)
    |- Target Asset: Production Mountain Roman Celtic Iron Ore (GUID: 5982)
    |- Target Asset: Production Mountain Roman Gold Ore (GUID: 50280)
    |- Target Asset: Production Mountain Roman Coal (GUID: 144810)
   

## Export to JSON & Image Export

Using the `save_to_json` method from the extractor. You can pass a custom `web_base_path` to fit your webapp asset structure. If left empty, it will provide the original image game path within the asset extractor.

The `IconProcessor` is a static utility class designed for batch processing, resizing, and converting `.dds` game icons into web-ready `.webp` files.

The `IconProcessor.export_icons()` method allows for either a Flattened structure (optimized for simple galleries) or a Mirrored structure (mimicking the game's internal directory hierarchy).

This is done by using wand + magick within the IconProcessor.

In [9]:
from assetextractor.conversion.statistics.icon_processor import IconProcessor

patron_list = list(patrons_data.values())

# Define base output folder
output_root = Path("results")

In [10]:
# =================================================================
# SCENARIO A: MIRRORED STRUCTURE (Mimics Game Folders)
# Use this for: Webapps that need the full "base/icon_content/..." tree.
# =================================================================
print(f" Scenario A: Mirrored ".center(80, "*"))
    
# 1. Export ALL physical files (Icons @ 128px, Big Portraits @ 512px, Small Portraits @ 128px)
# Recreates the game's directory hierarchy for everything.
extractor.export_all_patron_assets(
    output_base=output_root / "icons",
    quality=75,
    flatten=False        # Mirror hierarchy for standard assets & portraits
)

# 2. Save JSON with URLs like: "base/icon_content/religion/icon_name.webp"
extractor.save_to_json(
    file_path=output_root / "tables/patrons_en_original.json",
    web_base_path=None,  # No prefix, relative path from game tree root
    flatten=False        # Matches the mirrored file export paths
)

***************************** Scenario A: Mirrored *****************************
Exporting 40 standard icons (128x128)...
  [OK] -> icons\base\icon_content\religion\icon_2d_deity_neptune_0.webp
  [OK] -> icons\base\icon_content\infrastructure\icon_3d_resource_river_0.webp
  [OK] -> icons\base\icon_content\ornaments\shrines\icon_3d_shrine_neptune_0.webp
  [OK] -> icons\base\icon_content\religion\icon_2d_deity_minerva_0.webp
  [OK] -> icons\base\icon_content\infrastructure\icon_3d_stone_gate_0.webp
  [OK] -> icons\base\icon_content\ornaments\shrines\icon_3d_shrine_minerva_0.webp
  [OK] -> icons\base\icon_content\religion\icon_2d_deity_ceres_0.webp
  [OK] -> icons\base\icon_content\infrastructure\icon_3d_farm_fields_0.webp
  [OK] -> icons\base\icon_content\ornaments\shrines\icon_3d_shrine_ceres_0.webp
  [OK] -> icons\base\icon_content\religion\icon_2d_deity_cernunnos_0.webp
  [OK] -> icons\base\icon_content\infrastructure\icon_3d_resource_forest_0.webp
  [OK] -> icons\base\icon_content\or

In [11]:
# =================================================================
# SCENARIO B: FLATTENED STRUCTURE (Canonical Names)
# Use this for: Simple galleries where all icons are in one folder.
# =================================================================
print(f" Scenario B: Flattened ".center(80, "*"))

# 1. Export ALL physical files directly into the base target directory.
# Uses canonical asset names for standard assets and cached portrait filenames.
extractor.export_all_patron_assets(
    output_base=output_root / "icons_flat",
    quality=75,
    flatten=True         # Flattens folders out for icons & portraits
)

# 2. Save JSON with URLs like: "static/assets/icon_patron_mars.webp"
extractor.save_to_json(
    file_path=output_root / "tables/patrons_en_web.json",
    web_base_path="static/assets",  # Custom web URL prefix prepended to filenames
    flatten=True                    # Matches the flattened asset generation names
)

**************************** Scenario B: Flattened *****************************
Exporting 40 standard icons (128x128)...
  [OK] -> icons_flat\icon_patron_neptune.webp
  [OK] -> icons_flat\icon_effect_venilia_s_serenity.webp
  [OK] -> icons_flat\icon_mini_shrine_of_neptune.webp
  [OK] -> icons_flat\icon_patron_minerva.webp
  [OK] -> icons_flat\icon_effect_aegis_gorgoneion.webp
  [OK] -> icons_flat\icon_mini_shrine_of_minerva.webp
  [OK] -> icons_flat\icon_patron_ceres.webp
  [OK] -> icons_flat\icon_effect_vervactor_s_plough.webp
  [OK] -> icons_flat\icon_mini_shrine_of_ceres.webp
  [OK] -> icons_flat\icon_patron_cernunnos.webp
  [OK] -> icons_flat\icon_effect_arboreal_rhizome.webp
  [OK] -> icons_flat\icon_mini_shrine_of_cernunnos.webp
  [OK] -> icons_flat\icon_patron_mercury_lugus.webp
  [OK] -> icons_flat\icon_production_gold_mine_latium.webp
  [OK] -> icons_flat\icon_mini_shrine_of_mercury_lugus.webp
  [OK] -> icons_flat\icon_patron_mars.webp
  [OK] -> icons_flat\icon_land_gladiator